# VotingClassifier

Notebook for training and retraining the soft-voting ensemble.
Outputs are saved to `../notebook_outputs/` so the current app models in `artifacts/` stay untouched.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier

MODEL_NAME = "VotingClassifier"
TARGET_COLUMN = "target"
DATA_DIR = Path("../data")
ARTIFACTS_DIR = Path("../notebook_outputs")
WEIGHT_GRID = [
    [1, 1, 1, 1, 2, 2, 1],
    [1, 1, 1, 2, 2, 2, 1],
    [1, 2, 1, 1, 2, 2, 1],
]
BASE_CONFIGS = {
    "dt": {"max_depth": 4, "min_samples_split": 2, "min_samples_leaf": 4},
    "knn": {"n_neighbors": 4, "weights": "distance", "p": 2},
    "nb": {"var_smoothing": 1e-12},
    "rf": {"n_estimators": 100, "max_depth": 5, "min_samples_split": 6, "min_samples_leaf": 2},
    "ada": {"n_estimators": 150, "learning_rate": 0.3},
    "gb": {"n_estimators": 50, "learning_rate": 0.2, "max_depth": 2, "min_samples_split": 2, "min_samples_leaf": 1},
    "xgb": {"n_estimators": 80, "max_depth": 2, "learning_rate": 0.05},
}


def parse_numeric_value(value):
    if pd.isna(value):
        return value
    if isinstance(value, str):
        cleaned = value.strip()
        if cleaned.count(".") > 1:
            sign = -1 if cleaned.startswith("-") else 1
            digits_only = cleaned[1:] if sign == -1 else cleaned
            digits_only = digits_only.replace(".", "")
            return sign * float(f"0.{digits_only}")
        return float(cleaned)
    return float(value)


def load_dataset(file_path):
    df = pd.read_csv(file_path)
    object_columns = df.select_dtypes(include="object").columns
    for col in object_columns:
        df[col] = df[col].map(parse_numeric_value)
    return df


def split_xy(df):
    x = df.drop(columns=[TARGET_COLUMN]).copy()
    y = df[TARGET_COLUMN].copy()
    return x, y


def build_preprocessor(feature_names, scaler):
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", scaler),
        ]
    )
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, feature_names)],
        remainder="drop",
    )


def build_pipeline(estimator, feature_names, scaler):
    return Pipeline(
        steps=[
            ("preprocess", build_preprocessor(feature_names, scaler)),
            ("model", estimator),
        ]
    )


def build_voting_model(weights):
    estimators = [
        ("dt", build_pipeline(clone(DecisionTreeClassifier(random_state=42)).set_params(**BASE_CONFIGS["dt"]), feature_names, MinMaxScaler())),
        ("knn", build_pipeline(clone(KNeighborsClassifier()).set_params(**BASE_CONFIGS["knn"]), feature_names, StandardScaler())),
        ("nb", build_pipeline(clone(GaussianNB()).set_params(**BASE_CONFIGS["nb"]), feature_names, StandardScaler())),
        ("rf", build_pipeline(clone(RandomForestClassifier(random_state=42, n_jobs=-1)).set_params(**BASE_CONFIGS["rf"]), feature_names, MinMaxScaler())),
        ("ada", build_pipeline(clone(AdaBoostClassifier(random_state=42)).set_params(**BASE_CONFIGS["ada"]), feature_names, MinMaxScaler())),
        ("gb", build_pipeline(clone(GradientBoostingClassifier(random_state=42)).set_params(**BASE_CONFIGS["gb"]), feature_names, MinMaxScaler())),
        ("xgb", build_pipeline(clone(XGBClassifier(random_state=42, eval_metric="logloss", n_jobs=1)).set_params(**BASE_CONFIGS["xgb"]), feature_names, MinMaxScaler())),
    ]
    return VotingClassifier(estimators=estimators, voting="soft", weights=weights)


def get_scores(model, x):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(x)
    return None


def evaluate_model(model, x, y):
    y_pred = model.predict(x)
    y_score = get_scores(model, x)
    metrics = {
        "accuracy": float(accuracy_score(y, y_pred)),
        "precision": float(precision_score(y, y_pred, zero_division=0)),
        "recall": float(recall_score(y, y_pred, zero_division=0)),
        "f1": float(f1_score(y, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
    }
    if y_score is not None and len(np.unique(y)) > 1:
        metrics["roc_auc"] = float(roc_auc_score(y, y_score))
    return metrics


train_df = load_dataset(DATA_DIR / "train.csv")
val_df = load_dataset(DATA_DIR / "val.csv")
test_df = load_dataset(DATA_DIR / "test.csv")

x_train, y_train = split_xy(train_df)
x_val, y_val = split_xy(val_df)
x_test, y_test = split_xy(test_df)
feature_names = x_train.columns.tolist()

best_result = None

for weights in WEIGHT_GRID:
    model = build_voting_model(weights)
    model.fit(x_train, y_train)
    train_metrics = evaluate_model(model, x_train, y_train)
    val_metrics = evaluate_model(model, x_val, y_val)
    current_result = {
        "weights": weights,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "model": model,
    }
    if best_result is None:
        best_result = current_result
    else:
        current_key = (current_result["val_metrics"]["f1"], current_result["val_metrics"]["accuracy"])
        best_key = (best_result["val_metrics"]["f1"], best_result["val_metrics"]["accuracy"])
        if current_key > best_key:
            best_result = current_result

print("Best weights on validation:", best_result["weights"])
print("Train metrics:", json.dumps(best_result["train_metrics"], indent=2))
print("Validation metrics:", json.dumps(best_result["val_metrics"], indent=2))

x_train_val = pd.concat([x_train, x_val], ignore_index=True)
y_train_val = pd.concat([y_train, y_val], ignore_index=True)

final_model = build_voting_model(best_result["weights"])
final_model.fit(x_train_val, y_train_val)

train_val_metrics = evaluate_model(final_model, x_train_val, y_train_val)
test_metrics = evaluate_model(final_model, x_test, y_test)
test_predictions = final_model.predict(x_test)

models_dir = ARTIFACTS_DIR / "models"
reports_dir = ARTIFACTS_DIR / "reports"
predictions_dir = ARTIFACTS_DIR / "predictions"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / f"{MODEL_NAME}.joblib"
report_path = reports_dir / f"{MODEL_NAME}_metrics.json"
prediction_path = predictions_dir / f"{MODEL_NAME}_test_predictions.csv"

joblib.dump(final_model, model_path)

prediction_df = x_test.copy()
prediction_df["actual_target"] = y_test.values
prediction_df["predicted_target"] = test_predictions
prediction_df.to_csv(prediction_path, index=False)

report = {
    "model_name": MODEL_NAME,
    "base_configs": BASE_CONFIGS,
    "best_validation_weights": best_result["weights"],
    "train_metrics": best_result["train_metrics"],
    "validation_metrics": best_result["val_metrics"],
    "train_val_metrics_after_refit": train_val_metrics,
    "test_metrics": test_metrics,
    "saved_model_path": str(model_path),
    "prediction_path": str(prediction_path),
}

report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\nTrain+Val metrics after refit:")
print(json.dumps(train_val_metrics, indent=2))
print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))
print("\nSaved model:", model_path)
print("Saved report:", report_path)
print("Saved predictions:", prediction_path)

prediction_df.head()


Best weights on validation: [1, 1, 1, 1, 2, 2, 1]
Train metrics: {
  "accuracy": 0.9545454545454546,
  "precision": 0.9716981132075472,
  "recall": 0.9279279279279279,
  "f1": 0.9493087557603687,
  "confusion_matrix": [
    [
      128,
      3
    ],
    [
      8,
      103
    ]
  ],
  "roc_auc": 0.9913348462966783
}
Validation metrics: {
  "accuracy": 0.9333333333333333,
  "precision": 0.875,
  "recall": 1.0,
  "f1": 0.9333333333333333,
  "confusion_matrix": [
    [
      14,
      2
    ],
    [
      0,
      14
    ]
  ],
  "roc_auc": 0.9910714285714286
}

Train+Val metrics after refit:
{
  "accuracy": 0.9485294117647058,
  "precision": 0.9663865546218487,
  "recall": 0.92,
  "f1": 0.9426229508196722,
  "confusion_matrix": [
    [
      143,
      4
    ],
    [
      10,
      115
    ]
  ],
  "roc_auc": 0.9898775510204081
}

Test metrics:
{
  "accuracy": 0.8709677419354839,
  "precision": 0.8571428571428571,
  "recall": 0.8571428571428571,
  "f1": 0.8571428571428571,
  "confus

,age,trestbps,chol,thalach,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,actual_target,predicted_target
0,0.384303,-0.168240,-0.641646,-0.837597,0.107158,1.0,1.000000,0.0,1.0,1.0,0.5,1.0,1.0,1,1
1,-0.228879,-0.736870,-0.128635,0.106174,-0.891627,1.0,0.000000,0.0,1.0,0.0,0.0,0.0,0.0,0,0
2,0.829818,-0.545134,-0.357219,-0.175039,0.714629,1.0,0.666667,0.0,0.0,0.0,0.5,1.0,1.0,0,1
3,-0.395349,-0.545134,0.116827,-0.425278,-0.445445,0.0,0.666667,0.0,1.0,0.0,0.0,0.0,0.0,0,0
4,-0.139776,-0.623144,-0.186562,0.194515,-0.177735,1.0,0.666667,1.0,0.0,0.0,1.0,0.0,1.0,0,0
